# Sommelier Kaggle Wheels Dataset Builder

Notebook này chỉ dùng để tải Python wheels một lần rồi lưu thành private Kaggle Dataset. Notebook chạy pipeline chính sẽ Add Input dataset này và cài offline từ `/kaggle/input`, tránh tải lại requirements mỗi lần đổi GPU/restart.

Cách dùng:
1. Bật Internet cho notebook này.
2. Run All.
3. Save Version.
4. Tạo private Dataset từ thư mục output `sommelier_wheels`.
5. Add Dataset đó vào notebook chạy pipeline.

## 0. Cấu hình

Nếu đổi branch hoặc không muốn tải nhóm package nào, chỉnh các biến bên dưới trước khi chạy.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/lamkdhe180931-arch/sommelier.git"
BRANCH = "test-divide-stage"

WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "sommelier"
PIPELINE_DIR = REPO_DIR / "podcast-pipeline"
WHEEL_DIR = WORK_DIR / "sommelier_wheels"

DOWNLOAD_BASE_REQUIREMENTS = True
DOWNLOAD_TORCH_STACK = True
DOWNLOAD_NEMO_ASR = True
DOWNLOAD_SEPREFORMER_EXTRA_DEPS = False  # Để False nếu stage 3 dùng HF API, không dùng SepReformer local.
DOWNLOAD_CUDNN8_RUNTIME = True  # Dùng cho faster-whisper/ctranslate2 trong notebook 01.

TORCH_VERSION = "2.7.1"
TORCHAUDIO_VERSION = "2.7.1"
TORCHVISION_VERSION = "0.22.1"
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu126"

print("WHEEL_DIR:", WHEEL_DIR)

## 1. Helper chạy command

Dùng `subprocess.run` thay cho notebook magic `!` để notebook chạy ổn định hơn khi copy qua môi trường khác.

In [ ]:
import subprocess
import sys


def run_cmd(cmd, cwd=None):
    print("Running:", " ".join(str(x) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)

## 2. Clone repo và tạo `requirements-kaggle.txt`

Cell này loại `nemo-toolkit[all]` khỏi requirements gốc để tránh cài quá rộng. NeMo ASR sẽ được tải riêng ở cell sau.

In [ ]:
import os
from pathlib import Path

WORK_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    run_cmd(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR.name], cwd=WORK_DIR)
else:
    print("Repo already exists:", REPO_DIR)

req = (PIPELINE_DIR / "requirements.txt").read_text(encoding="utf-8")
filtered = []
for line in req.splitlines():
    stripped = line.strip()
    if not stripped:
        continue
    if "nemo-toolkit[all]" in stripped:
        continue
    filtered.append(line)

requirements_kaggle = PIPELINE_DIR / "requirements-kaggle.txt"
requirements_kaggle.write_text("\n".join(filtered) + "\n", encoding="utf-8")
print(requirements_kaggle.read_text(encoding="utf-8"))


## 3. Tải wheels cho requirements chính

Output sẽ nằm trong `/kaggle/working/sommelier_wheels`.

In [ ]:
import shutil

WHEEL_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(PIPELINE_DIR / "requirements-kaggle.txt", WHEEL_DIR / "requirements-kaggle.txt")

if DOWNLOAD_BASE_REQUIREMENTS:
    run_cmd([
        sys.executable, "-m", "pip", "download",
        "-r", PIPELINE_DIR / "requirements-kaggle.txt",
        "-d", WHEEL_DIR,
    ])
else:
    print("DOWNLOAD_BASE_REQUIREMENTS=False, skip.")

## 4. Tải torch CUDA stack

Notebook pipeline hiện dùng torch/torchaudio/torchvision CUDA riêng. Tải nhóm này vào cùng wheel dataset để cài offline.

In [ ]:
if DOWNLOAD_TORCH_STACK:
    run_cmd([
        sys.executable, "-m", "pip", "download",
        f"torch=={TORCH_VERSION}",
        f"torchaudio=={TORCHAUDIO_VERSION}",
        f"torchvision=={TORCHVISION_VERSION}",
        "--index-url", TORCH_INDEX_URL,
        "-d", WHEEL_DIR,
    ])
else:
    print("DOWNLOAD_TORCH_STACK=False, skip.")

## 5. Tải NeMo ASR stack

Chỉ cần nếu notebook pipeline vẫn dùng Sortformer/NeMo cho diarization.

In [ ]:
if DOWNLOAD_NEMO_ASR:
    run_cmd([
        sys.executable, "-m", "pip", "download",
        "lightning==2.4.0",
        "pytorch-lightning==2.5.2",
        "nemo-toolkit[asr]==2.4.0",
        "-d", WHEEL_DIR,
    ])
else:
    print("DOWNLOAD_NEMO_ASR=False, skip.")

## 6. Tải deps phụ của SepReformer nếu còn dùng local SepReformer

Nếu stage 3 chuyển sang Hugging Face API thì để `DOWNLOAD_SEPREFORMER_EXTRA_DEPS=False`.

In [ ]:
if DOWNLOAD_SEPREFORMER_EXTRA_DEPS:
    run_cmd([
        sys.executable, "-m", "pip", "download",
        "mir-eval==0.7",
        "ptflops==0.7.4",
        "thop==0.1.1.post2209072238",
        "torchinfo==1.8.0",
        "-d", WHEEL_DIR,
    ])
else:
    print("DOWNLOAD_SEPREFORMER_EXTRA_DEPS=False, skip.")

## 7. Tải cuDNN8 runtime cho faster-whisper/ctranslate2

Notebook pipeline `01_v01_full_hf_api_overlap_offline_wheels.ipynb` cài package này offline vào `/kaggle/working/cudnn8` trước stage ASR.

In [ ]:
if DOWNLOAD_CUDNN8_RUNTIME:
    run_cmd([
        sys.executable, "-m", "pip", "download",
        "nvidia-cudnn-cu12==8.9.7.29",
        "-d", WHEEL_DIR,
    ])
else:
    print("DOWNLOAD_CUDNN8_RUNTIME=False, skip.")

## 8. Kiểm tra output trước khi Save Version

Sau cell này, bấm `Save Version`. Khi version chạy xong, tạo private Dataset từ thư mục output `sommelier_wheels`.

In [ ]:
files = sorted(WHEEL_DIR.glob("*"))
print("Wheel/cache files:", len(files))

size_result = subprocess.run(["du", "-sh", str(WHEEL_DIR)], check=False, capture_output=True, text=True)
print(size_result.stdout.strip())

print("\nFirst 80 files:")
for p in files[:80]:
    print(p.name)

print("\nSau khi Save Version, tạo private Kaggle Dataset từ folder:")
print(WHEEL_DIR)


## 9. Cài offline trong notebook pipeline chính

Sau khi tạo private Dataset và Add Input vào notebook chính, dùng cell mẫu này ở notebook pipeline:

```python
from pathlib import Path
import subprocess
import sys

WHEEL_DIR = Path("/kaggle/input/sommelier-wheels")
if not WHEEL_DIR.exists():
    raise FileNotFoundError(f"Không thấy wheel dataset: {WHEEL_DIR}")

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-index", f"--find-links={WHEEL_DIR}",
    "-r", str(WHEEL_DIR / "requirements-kaggle.txt"),
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-index", f"--find-links={WHEEL_DIR}",
    "torch==2.7.1", "torchaudio==2.7.1", "torchvision==0.22.1",
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-index", f"--find-links={WHEEL_DIR}",
    "lightning==2.4.0", "pytorch-lightning==2.5.2", "nemo-toolkit[asr]==2.4.0",
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-index", f"--find-links={WHEEL_DIR}",
    "--target", "/kaggle/working/cudnn8",
    "nvidia-cudnn-cu12==8.9.7.29",
], check=True)
```